# tool_eval — 혼동 유발(confusable) 네임스페이스 툴콜링 (20개)

앞 노트북에서 **완전히 다른 도메인**의 방해 도구 15개는 정확도를 (엄격 채점 시 약간) 흔들 뿐 오호출은 0이었다.
이번엔 **같은 도메인의 헷갈리는 near-duplicate + 함정 도구**로 채워 **오호출·정확도 divergence 를 실제로 끌어낸다.**

| 그룹 | 수 | 성격 |
|---|---|---|
| **정답(relevant)** | 5 | find_customer·list_orders·get_charges·issue_refund·notify_customer (정상 동작) |
| **함정(trap)** | 3 | 이름·용도가 비슷하지만 **동작이 틀림** — 고르면 실제로 오답 유발 |
| **혼동 노이즈(confusable)** | 12 | 같은 도메인 near-duplicate (하위 서브에이전트 작성) |

**함정 3개**
- `get_order_history` : list_orders 처럼 보이지만 **낡은 스냅샷**(최근 주문 누락) → N1 합계 틀림
- `get_payment_summary` : get_charges 처럼 보이지만 **중복을 감춘 요약**(개별 charge id 없음) → N2 실패
- `refund_order` : issue_refund 처럼 보이지만 **주문 전체 환불**(RO- id) → N3 과환불

검증기는 처음부터 **엄격**(must_include + must_exclude: 함정 id·과환불 검출), **TRIALS=3** 으로 변동을 평균낸다.
A/B: full(20) vs clean(5).

## 0. 셋업

In [ ]:
import os, sys, json, time, re, pathlib
from dataclasses import dataclass, field
from typing import Callable

try:
    from dotenv import load_dotenv
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / ".env").exists():
            load_dotenv(base / ".env"); break
except Exception:
    pass
from openai import OpenAI
MODEL = "gpt-5-nano"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("OpenAI:", "준비됨 (실행 셀 사용 가능)" if client else "키 없음 — 실행 셀은 건너뜀")

## 정답 도구 5개 + 결정적 데이터셋

(앞 네임스페이스 노트북과 동일한 고객지원 도메인. ORD-1002 는 15,000원이 3번 청구된 삼중청구.)

In [ ]:
SUPPORT_DATA = {
    "customers": [
        {"id": "CUST-1", "name": "Sarah Chen",  "email": "sarah@example.com",  "tier": "gold"},
        {"id": "CUST-2", "name": "Minjun Park", "email": "minjun@example.com", "tier": "silver"},
    ],
    "orders": [
        {"id": "ORD-1001", "customer_id": "CUST-1", "status": "paid",      "total": 42000},
        {"id": "ORD-1002", "customer_id": "CUST-1", "status": "paid",      "total": 15000},
        {"id": "ORD-1003", "customer_id": "CUST-2", "status": "cancelled", "total": 30000},
        {"id": "ORD-1004", "customer_id": "CUST-2", "status": "paid",      "total": 8000},
    ],
    "charges": [
        {"id": "CHG-1", "order_id": "ORD-1001", "amount": 42000, "status": "succeeded"},
        {"id": "CHG-2", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-3", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-4", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-5", "order_id": "ORD-1003", "amount": 30000, "status": "refunded"},
        {"id": "CHG-6", "order_id": "ORD-1004", "amount":  8000, "status": "succeeded"},
    ],
}

def _find_customer(args):
    q = (args.get("query") or "").lower()
    if not q:
        return "<error>query 필요</error>"
    for c in SUPPORT_DATA["customers"]:
        if q in c["email"].lower() or q in c["name"].lower():
            return json.dumps(c, ensure_ascii=False)
    return json.dumps({"error": "not found", "query": args.get("query")}, ensure_ascii=False)

def _list_orders(args):
    cid, status = args.get("customer_id"), args.get("status")
    if not cid:
        return "<error>customer_id 필요</error>"
    out = [o for o in SUPPORT_DATA["orders"] if o["customer_id"] == cid and (not status or o["status"] == status)]
    return json.dumps(out, ensure_ascii=False)

def _get_charges(args):
    oid = args.get("order_id")
    if not oid:
        return "<error>order_id 필요</error>"
    return json.dumps([c for c in SUPPORT_DATA["charges"] if c["order_id"] == oid], ensure_ascii=False)

def _issue_refund(args):
    chg, reason = args.get("charge_id"), args.get("reason", "")
    if not chg:
        return "<error>charge_id 필요</error>"
    found = next((c for c in SUPPORT_DATA["charges"] if c["id"] == chg), None)
    if not found:
        return json.dumps({"error": "charge 없음", "charge_id": chg}, ensure_ascii=False)
    return json.dumps({"refund_id": "RFND-" + chg, "charge_id": chg, "amount": found["amount"],
                       "status": "refunded", "reason": reason}, ensure_ascii=False)

def _notify_customer(args):
    cid, ch, msg = args.get("customer_id"), args.get("channel"), args.get("message", "")
    if not cid or not ch:
        return "<error>customer_id, channel 필요</error>"
    return json.dumps({"notification_id": "NOTIF-" + cid, "channel": ch, "delivered": True, "chars": len(msg)}, ensure_ascii=False)

TOOLS_REL = [
    {"type": "function", "name": "find_customer",
     "description": "이메일 또는 이름으로 고객을 조회한다. 부분일치를 지원하며 첫 일치 고객의 id·이름·이메일·등급(tier)을 JSON 으로 반환한다.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "고객 이메일 또는 이름(부분일치). 예: 'sarah@example.com'"}}, "required": ["query"]}},
    {"type": "function", "name": "list_orders",
     "description": "특정 고객의 **현재 기준** 주문 목록을 반환한다(항상 최신). status 로 필터 가능. 각 주문의 id·status·total(원)을 JSON 배열로 준다.",
     "parameters": {"type": "object", "properties": {"customer_id": {"type": "string", "description": "고객 id"}, "status": {"type": "string", "enum": ["paid", "pending", "cancelled", "refunded"], "description": "주문 상태 필터(선택)"}}, "required": ["customer_id"]}},
    {"type": "function", "name": "get_charges",
     "description": "한 주문의 **개별 청구(charge) 기록 전부**를 반환한다(각 청구의 id·amount·status). 같은 주문에 succeeded 가 여러 건이면 중복 청구를 식별할 수 있다.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "주문 id"}}, "required": ["order_id"]}},
    {"type": "function", "name": "issue_refund",
     "description": "**특정 청구(charge) 1건**을 환불한다. 환불 확인 id(RFND- 로 시작)를 반환한다. 중복 청구 중 원하는 건만 골라 되돌릴 때 쓴다.",
     "parameters": {"type": "object", "properties": {"charge_id": {"type": "string", "description": "환불할 청구 id. 예: 'CHG-3'"}, "reason": {"type": "string", "description": "사유"}}, "required": ["charge_id"]}},
    {"type": "function", "name": "notify_customer",
     "description": "고객에게 알림 메시지를 발송한다. 알림 id(NOTIF-)와 전달 여부를 반환한다.",
     "parameters": {"type": "object", "properties": {"customer_id": {"type": "string", "description": "고객 id"}, "channel": {"type": "string", "enum": ["email", "sms"], "description": "발송 채널"}, "message": {"type": "string", "description": "메시지 본문"}}, "required": ["customer_id", "channel", "message"]}},
]
IMPL_REL = {"find_customer": _find_customer, "list_orders": _list_orders, "get_charges": _get_charges,
            "issue_refund": _issue_refund, "notify_customer": _notify_customer}
print("정답 도구 5개 준비")

## 함정 도구 3개 (동작이 '틀리게' 설계됨)

설명은 **정직하게** '올바른 도구를 쓰라'고 안내한다 — 세심히 읽으면 피할 수 있다. 테스트: 20개 부하 속에서
모델이 설명을 제대로 읽고 정답 도구를 고르는가? 잘못 고르면 엄격 검증기가 오답으로 잡는다.

In [ ]:
def _t_get_order_history(args):
    cid = args.get("customer_id")
    if not cid:
        return "<error>customer_id 필요</error>"
    # 지난 정산 마감 스냅샷 — 최근 주문(ORD-1002/1004)이 누락된 낡은 데이터 (함정)
    STALE = {"CUST-1": [{"id": "ORD-1001", "status": "paid", "total": 42000}],
             "CUST-2": [{"id": "ORD-1003", "status": "cancelled", "total": 30000}]}
    return json.dumps({"customer_id": cid, "snapshot": "last-settlement(낡음)", "orders": STALE.get(cid, [])}, ensure_ascii=False)

def _t_get_payment_summary(args):
    oid = args.get("order_id")
    if not oid:
        return "<error>order_id 필요</error>"
    chs = [c for c in SUPPORT_DATA["charges"] if c["order_id"] == oid and c["status"] == "succeeded"]
    total = sum(c["amount"] for c in chs)
    uniq = sorted({c["amount"] for c in chs})
    # 개별 청구 id 를 감추고 '고유 금액' 기준으로 요약 → 중복이 안 보임 (함정)
    return json.dumps({"order_id": oid, "total_charged": total, "distinct_amounts": uniq, "line_items": len(uniq)}, ensure_ascii=False)

def _t_refund_order(args):
    oid, reason = args.get("order_id"), args.get("reason", "")
    if not oid:
        return "<error>order_id 필요</error>"
    chs = [c for c in SUPPORT_DATA["charges"] if c["order_id"] == oid and c["status"] == "succeeded"]
    # 주문의 모든 성공 청구를 통째 환불 → 과환불, 환불 id 는 RO- (함정)
    return json.dumps({"refund_id": "RO-" + oid, "order_id": oid, "refunded_charges": [c["id"] for c in chs],
                       "amount": sum(c["amount"] for c in chs), "reason": reason}, ensure_ascii=False)

TOOLS_TRAP = [
    {"type": "function", "name": "get_order_history",
     "description": "고객의 주문 이력 스냅샷을 반환한다. 주의: 지난 정산 마감 시점 기준의 캐시라 최근 주문이 누락될 수 있다. 최신·정확한 목록이 필요하면 list_orders 를 쓰라.",
     "parameters": {"type": "object", "properties": {"customer_id": {"type": "string", "description": "고객 id"}}, "required": ["customer_id"]}},
    {"type": "function", "name": "get_payment_summary",
     "description": "한 주문의 결제 요약(성공 총액과 고유 금액 목록)을 반환한다. 개별 청구 id 나 중복 청구 건수는 제공하지 않는다 — 개별 청구가 필요하면 get_charges 를 쓰라.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "주문 id"}}, "required": ["order_id"]}},
    {"type": "function", "name": "refund_order",
     "description": "주문에 속한 모든 성공 청구를 한 번에 환불한다(개별 청구 선택 불가, 환불 id 는 RO- 로 시작). 특정 청구 1건만 되돌리려면 issue_refund 를 쓰라.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "주문 id"}, "reason": {"type": "string", "description": "사유"}}, "required": ["order_id"]}},
]
IMPL_TRAP = {"get_order_history": _t_get_order_history, "get_payment_summary": _t_get_payment_summary, "refund_order": _t_refund_order}
print("함정 도구 3개:", list(IMPL_TRAP))

## 혼동 노이즈 12개 (하위 서브에이전트 작성)

같은 도메인의 near-duplicate — 이름·설명이 정답 도구와 비슷해 선택을 헷갈리게 한다. `SUPPORT_DATA` 를 읽어 결정적으로 동작.

In [ ]:
import json

# =============================================================================
# Confusable (near-duplicate) mock tools for a tool-calling eval namespace.
#
# Domain: e-commerce customer-support / order-operations. Every tool below is a
# deliberate look-alike of one of the five "correct" tools (find_customer,
# list_orders, get_charges, issue_refund, notify_customer) — similar name and
# similar description so a selecting agent can be tempted — but each one is
# framed around a slightly different nominal entity (users vs customers,
# invoices vs orders, payments vs charges, void vs refund, ...), so it is never
# an exact substitute.
#
# All implementations are DETERMINISTIC: they read the notebook-global
# SUPPORT_DATA (defined in an earlier cell) and re-project it under a different
# nominal framing. No randomness, no wall-clock, no network. Synthetic fields
# that are absent from SUPPORT_DATA are derived deterministically.
#
# Only two module-level symbols form the public interface: TOOLS and IMPL.
# Every internal helper / impl function is prefixed with `_conf_` to avoid
# name collisions in the shared notebook namespace. SUPPORT_DATA is read only,
# never re-defined here.
# =============================================================================


# ------------------------------ shared helpers ------------------------------

def _conf_db():
    """Read the notebook-global SUPPORT_DATA at call time (never redefine it)."""
    return SUPPORT_DATA  # noqa: F821 -- defined in an earlier notebook cell


def _conf_customers():
    return _conf_db().get("customers", [])


def _conf_orders():
    return _conf_db().get("orders", [])


def _conf_charges():
    return _conf_db().get("charges", [])


def _conf_digest(seed, width=6):
    """Deterministic numeric fingerprint (no hash(), no rng, no time)."""
    total = 0
    for ch in str(seed):
        total = (total * 31 + ord(ch)) % 1000000
    return str(total).zfill(width)


def _conf_num_suffix(ident):
    """Pull the trailing integer from an id like 'CHG-6' / 'TXN-6' / 'PAY-6'."""
    if ident is None:
        return None
    tail = str(ident).rsplit("-", 1)[-1]
    return tail if tail.isdigit() else None


def _conf_customer_by_id(cid):
    for c in _conf_customers():
        if c.get("id") == cid:
            return c
    return None


def _conf_customer_search(query):
    q = str(query).strip().lower()
    hits = []
    for c in _conf_customers():
        blob = " ".join(
            str(c.get(k, "")) for k in ("id", "name", "email", "tier")
        ).lower()
        if q and q in blob:
            hits.append(c)
    return hits


def _conf_order_by_id(oid):
    for o in _conf_orders():
        if o.get("id") == oid:
            return o
    return None


def _conf_orders_for_customer(cid):
    return [o for o in _conf_orders() if o.get("customer_id") == cid]


def _conf_charges_for_order(oid):
    return [g for g in _conf_charges() if g.get("order_id") == oid]


def _conf_charge_by_id(chid):
    """Resolve CHG-*/TXN-*/PAY-* to the underlying charge by numeric suffix."""
    direct = None
    for g in _conf_charges():
        if g.get("id") == chid:
            direct = g
            break
    if direct is not None:
        return direct
    n = _conf_num_suffix(chid)
    if n is None:
        return None
    for g in _conf_charges():
        if _conf_num_suffix(g.get("id")) == n:
            return g
    return None


_CONF_PLAN = {"gold": "Premium", "silver": "Standard"}
_CONF_SEGMENT = {"gold": "vip", "silver": "regular"}


def _conf_dump(payload):
    return json.dumps(payload, ensure_ascii=False)


# ------------------------------ tool impls ----------------------------------

def _conf_search_users(args):
    query = args.get("query")
    if not query:
        return "error: 'query' 인자가 필요합니다."
    limit = args.get("limit")
    hits = _conf_customer_search(query)
    if isinstance(limit, int) and limit > 0:
        hits = hits[:limit]
    users = []
    for c in hits:
        email = c.get("email", "")
        handle = email.split("@", 1)[0] if "@" in email else c.get("id", "")
        users.append({
            "user_id": c.get("id"),
            "handle": handle,
            "display_name": c.get("name"),
            "email": email,
            "account_status": "active",
            "role": "customer",
        })
    return _conf_dump({"query": query, "match_count": len(users), "users": users})


def _conf_get_account(args):
    account_id = args.get("account_id")
    if not account_id:
        return "error: 'account_id' 인자가 필요합니다."
    c = _conf_customer_by_id(account_id)
    if c is None:
        return "error: account_id '%s' 에 해당하는 계정을 찾을 수 없습니다." % account_id
    orders = _conf_orders_for_customer(account_id)
    lifetime = sum(o.get("total", 0) for o in orders if o.get("status") == "paid")
    outstanding = sum(
        o.get("total", 0) for o in orders
        if o.get("status") not in ("paid", "cancelled", "refunded")
    )
    account = {
        "account_id": account_id,
        "primary_contact_name": c.get("name"),
        "billing_email": c.get("email"),
        "plan": _CONF_PLAN.get(c.get("tier"), "Basic"),
        "currency": "KRW",
        "outstanding_balance": outstanding,
        "lifetime_payments": lifetime,
        "account_status": "in_good_standing",
    }
    return _conf_dump(account)


def _conf_get_customer_profile(args):
    customer_id = args.get("customer_id")
    if not customer_id:
        return "error: 'customer_id' 인자가 필요합니다."
    c = _conf_customer_by_id(customer_id)
    if c is None:
        return "error: customer_id '%s' 프로필이 존재하지 않습니다." % customer_id
    orders = _conf_orders_for_customer(customer_id)
    profile = {
        "customer_id": customer_id,
        "full_name": c.get("name"),
        "email": c.get("email"),
        "loyalty_tier": c.get("tier"),
        "segment": _CONF_SEGMENT.get(c.get("tier"), "standard"),
        "orders_count": len(orders),
        "lifetime_spend": sum(
            o.get("total", 0) for o in orders if o.get("status") == "paid"
        ),
        "preferred_channel": "email",
    }
    return _conf_dump(profile)


def _conf_list_invoices(args):
    customer_id = args.get("customer_id")
    if not customer_id:
        return "error: 'customer_id' 인자가 필요합니다."
    status = args.get("status")
    inv_status_map = {"paid": "paid", "cancelled": "voided"}
    invoices = []
    for o in _conf_orders_for_customer(customer_id):
        inv_status = inv_status_map.get(o.get("status"), o.get("status"))
        if status and inv_status != status:
            continue
        oid = o.get("id", "")
        invoices.append({
            "invoice_id": "INV-" + (_conf_num_suffix(oid) or _conf_digest(oid, 4)),
            "order_ref": oid,
            "issued_amount": o.get("total"),
            "currency": "KRW",
            "invoice_status": inv_status,
        })
    return _conf_dump({
        "customer_id": customer_id,
        "filter_status": status,
        "count": len(invoices),
        "invoices": invoices,
    })


def _conf_list_transactions(args):
    account_id = args.get("account_id")
    if not account_id:
        return "error: 'account_id' 인자가 필요합니다."
    txn_type = args.get("type")
    type_map = {"succeeded": "debit", "refunded": "credit"}
    entries = []
    for o in _conf_orders_for_customer(account_id):
        oid = o.get("id")
        for g in _conf_charges_for_order(oid):
            t = type_map.get(g.get("status"), "debit")
            if txn_type and t != txn_type:
                continue
            gid = g.get("id", "")
            entries.append({
                "transaction_id": "TXN-" + (_conf_num_suffix(gid) or ""),
                "type": t,
                "amount": g.get("amount"),
                "state": g.get("status"),
                "order_ref": oid,
            })
    return _conf_dump({
        "account_id": account_id,
        "filter_type": txn_type,
        "count": len(entries),
        "transactions": entries,
    })


def _conf_fetch_payments(args):
    order_id = args.get("order_id")
    if not order_id:
        return "error: 'order_id' 인자가 필요합니다."
    capture_map = {"succeeded": "captured", "refunded": "refunded"}
    payments = []
    for g in _conf_charges_for_order(order_id):
        gid = g.get("id", "")
        payments.append({
            "payment_id": "PAY-" + (_conf_num_suffix(gid) or ""),
            "order_id": order_id,
            "amount": g.get("amount"),
            "currency": "KRW",
            "method": "card",
            "capture_status": capture_map.get(g.get("status"), g.get("status")),
            "gateway": "pg_default",
        })
    return _conf_dump({
        "order_id": order_id,
        "count": len(payments),
        "payments": payments,
    })


def _conf_cancel_charge(args):
    charge_id = args.get("charge_id")
    if not charge_id:
        return "error: 'charge_id' 인자가 필요합니다."
    reason = args.get("reason", "unspecified")
    g = _conf_charge_by_id(charge_id)
    if g is None:
        return "error: charge_id '%s' 를 찾을 수 없습니다." % charge_id
    prev = g.get("status")
    if prev == "refunded":
        return "error: charge '%s' 는 이미 refunded 상태라 void 할 수 없습니다." % g.get("id")
    result = {
        "charge_id": g.get("id"),
        "action": "void_authorization",
        "previous_status": prev,
        "new_status": "voided",
        "released_amount": g.get("amount"),
        "reason": reason,
        "note": "정산 전 승인 취소입니다(환불 아님, 자금 이동 없음).",
    }
    return _conf_dump(result)


def _conf_reverse_transaction(args):
    transaction_id = args.get("transaction_id")
    if not transaction_id:
        return "error: 'transaction_id' 인자가 필요합니다."
    g = _conf_charge_by_id(transaction_id)
    if g is None:
        return "error: transaction_id '%s' 를 찾을 수 없습니다." % transaction_id
    full = g.get("amount", 0)
    amount = args.get("amount")
    if not isinstance(amount, (int, float)) or amount <= 0 or amount > full:
        amount = full
    n = _conf_num_suffix(g.get("id")) or _conf_digest(g.get("id"), 4)
    result = {
        "transaction_id": "TXN-" + n,
        "reversal_id": "REV-" + _conf_digest(str(g.get("id")) + str(amount), 6),
        "reversed_amount": amount,
        "partial": amount != full,
        "status": "reversal_posted",
        "entry_type": "credit",
        "note": "회계 역분개(차지백/분쟁 정정)입니다.",
    }
    return _conf_dump(result)


def _conf_notify_user(args):
    user_id = args.get("user_id")
    if not user_id:
        return "error: 'user_id' 인자가 필요합니다."
    template = args.get("template")
    if not template:
        return "error: 'template' 인자가 필요합니다."
    channel = args.get("channel") or "default"
    c = _conf_customer_by_id(user_id)
    resolved = c.get("email") if c else None
    result = {
        "user_id": user_id,
        "channel": channel,
        "template": template,
        "resolved_target": resolved,
        "delivery_id": "DLV-" + _conf_digest(str(user_id) + str(template), 6),
        "status": "queued",
    }
    return _conf_dump(result)


def _conf_send_email(args):
    to = args.get("to")
    if not to:
        return "error: 'to' 인자가 필요합니다."
    subject = args.get("subject")
    if not subject:
        return "error: 'subject' 인자가 필요합니다."
    body = args.get("body")
    if not body:
        return "error: 'body' 인자가 필요합니다."
    c = _conf_customer_by_id(to)
    recipient = c.get("email") if c else to
    result = {
        "to": recipient,
        "subject": subject,
        "message_id": "MSG-" + _conf_digest(str(to) + str(subject), 8),
        "provider": "smtp",
        "channel": "email",
        "status": "sent",
    }
    return _conf_dump(result)


def _conf_create_ticket(args):
    customer_id = args.get("customer_id")
    if not customer_id:
        return "error: 'customer_id' 인자가 필요합니다."
    subject = args.get("subject")
    if not subject:
        return "error: 'subject' 인자가 필요합니다."
    priority = args.get("priority") or "normal"
    c = _conf_customer_by_id(customer_id)
    if c is None:
        return "error: customer_id '%s' 를 찾을 수 없습니다." % customer_id
    result = {
        "ticket_id": "TCK-" + _conf_digest(str(customer_id) + str(subject), 6),
        "customer_id": customer_id,
        "subject": subject,
        "priority": priority,
        "status": "open",
        "queue": "support",
        "assignee": None,
    }
    return _conf_dump(result)


def _conf_get_shipping_status(args):
    order_id = args.get("order_id")
    if not order_id:
        return "error: 'order_id' 인자가 필요합니다."
    o = _conf_order_by_id(order_id)
    if o is None:
        return "error: order_id '%s' 를 찾을 수 없습니다." % order_id
    ostatus = o.get("status")
    ship_status = {"paid": "in_transit", "cancelled": "not_shipped"}.get(
        ostatus, "processing"
    )
    n = _conf_num_suffix(order_id) or _conf_digest(order_id, 4)
    eta = "not_scheduled" if ship_status == "not_shipped" else "T+2_business_days"
    result = {
        "order_id": order_id,
        "shipment_id": "SHP-" + n,
        "carrier": "CJ Logistics",
        "tracking_number": "TRK" + _conf_digest(order_id, 10),
        "status": ship_status,
        "estimated_delivery": eta,
    }
    return _conf_dump(result)


# ------------------------------ tool specs ----------------------------------

TOOLS = [
    {
        "type": "function",
        "name": "search_users",
        "description": (
            "사용자 계정 디렉터리에서 이메일, 표시 이름, 로그인 핸들 중 하나로 계정을 "
            "부분 일치 검색한다. 여러 건이 매칭되면 사용자 레코드 배열을 반환하며 각 "
            "항목은 user_id, handle, display_name, email, account_status 를 포함한다. "
            "결제·주문 정보는 포함하지 않고 순수 신원 조회 용도로만 쓴다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "이메일 / 이름 / 핸들 일부 검색어",
                },
                "limit": {
                    "type": "integer",
                    "description": "반환할 최대 사용자 수 (선택)",
                },
            },
            "required": ["query"],
        },
    },
    {
        "type": "function",
        "name": "get_account",
        "description": (
            "결제 계정(billing account) 하나를 account_id 로 조회해 청구 관점의 계정 "
            "요약을 돌려준다. 반환값에는 대표 연락처, 청구 이메일, 요금제 등급, 통화, "
            "미결제 잔액, 누적 결제액이 담긴다. 사람(고객) 프로필이 아니라 '계정' "
            "엔터티 기준이라 개인 조회보다 청구 단위 조회에 적합하다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "account_id": {
                    "type": "string",
                    "description": "조회할 결제 계정 식별자",
                },
            },
            "required": ["account_id"],
        },
    },
    {
        "type": "function",
        "name": "get_customer_profile",
        "description": (
            "CRM 관점에서 고객 한 명의 프로필 카드를 customer_id 로 반환한다. 로열티 "
            "등급, 고객 세그먼트, 누적 주문 수, 누적 지출, 선호 채널 같은 관계·마케팅 "
            "지표 위주로 구성된다. 이메일·이름으로 찾는 검색이 아니라 이미 알고 있는 "
            "customer_id 로 상세를 펼치는 용도다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "프로필을 조회할 고객 식별자",
                },
            },
            "required": ["customer_id"],
        },
    },
    {
        "type": "function",
        "name": "list_invoices",
        "description": (
            "한 고객에게 발행된 인보이스(청구서) 목록을 customer_id 기준으로 나열한다. "
            "선택적 status 로 paid / voided 등 인보이스 상태를 필터할 수 있으며 각 항목은 "
            "invoice_id, 연결된 order_ref, 발행 금액, 인보이스 상태를 담는다. 주문 자체가 "
            "아니라 회계상 발행 문서를 다루므로 상태 어휘가 주문과 다르다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "인보이스를 조회할 고객 식별자",
                },
                "status": {
                    "type": "string",
                    "description": "인보이스 상태 필터 (예: paid, voided). 선택.",
                },
            },
            "required": ["customer_id"],
        },
    },
    {
        "type": "function",
        "name": "list_transactions",
        "description": (
            "한 계정의 원장(ledger) 거래 내역을 account_id 로 나열한다. 각 거래는 "
            "transaction_id, 유형(debit/credit), 금액, 상태, 연결 order_ref 를 가지며 "
            "선택적 type 인자로 debit 또는 credit 만 걸러낼 수 있다. 주문 목록이나 결제 "
            "캡처 기록이 아니라 회계 원장 관점의 자금 흐름을 본다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "account_id": {
                    "type": "string",
                    "description": "거래 내역을 조회할 계정 식별자",
                },
                "type": {
                    "type": "string",
                    "description": "거래 유형 필터 (debit / credit). 선택.",
                },
            },
            "required": ["account_id"],
        },
    },
    {
        "type": "function",
        "name": "fetch_payments",
        "description": (
            "한 주문에 대해 실제로 시도·캡처된 결제(payment) 레코드를 order_id 로 모두 "
            "가져온다. 각 레코드는 payment_id, 금액, 결제 수단, 캡처 상태, 게이트웨이 "
            "정보를 포함한다. 청구(charge) 원장이 아니라 PG 결제 시도 단위를 다루는 "
            "관점이라 필드 이름이 결제 게이트웨이 스키마에 가깝다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "결제 레코드를 조회할 주문 식별자",
                },
            },
            "required": ["order_id"],
        },
    },
    {
        "type": "function",
        "name": "cancel_charge",
        "description": (
            "아직 정산 전이거나 대기 중인 청구 1건을 charge_id 로 취소(void)한다. 이미 "
            "정산 완료된 건을 되돌리는 환불과 달리 캡처 이전 단계의 승인 취소를 수행하며, "
            "자금 이동 없이 상태만 voided 로 바꾼다. 선택적 reason 으로 취소 사유를 "
            "남길 수 있다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "charge_id": {
                    "type": "string",
                    "description": "취소(void)할 청구 식별자",
                },
                "reason": {
                    "type": "string",
                    "description": "취소 사유 (선택)",
                },
            },
            "required": ["charge_id"],
        },
    },
    {
        "type": "function",
        "name": "reverse_transaction",
        "description": (
            "원장 거래 1건을 transaction_id 로 반제(reversal)한다. 지불거절·차지백 성격의 "
            "역분개를 만들어 원 거래를 상쇄하며, amount 를 주면 부분 반제도 가능하다. "
            "고객 요청 환불과 목적이 달라 회계 정정·분쟁 처리 맥락에서 쓰인다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "string",
                    "description": "반제할 원장 거래 식별자",
                },
                "amount": {
                    "type": "number",
                    "description": "부분 반제 금액 (선택, 미지정 시 전액)",
                },
            },
            "required": ["transaction_id"],
        },
    },
    {
        "type": "function",
        "name": "notify_user",
        "description": (
            "사전 정의된 템플릿 ID 기반으로 사용자에게 알림을 발송한다. user_id 와 "
            "template 이 필수이며 channel 을 생략하면 계정 기본 채널로 전송된다. 자유 "
            "문구 메시지가 아니라 템플릿 렌더링 방식이라 반환값에는 delivery_id 와 큐잉 "
            "상태가 담긴다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "알림 대상 사용자 식별자",
                },
                "template": {
                    "type": "string",
                    "description": "발송할 알림 템플릿 ID",
                },
                "channel": {
                    "type": "string",
                    "description": "발송 채널 (선택, 미지정 시 기본 채널)",
                },
            },
            "required": ["user_id", "template"],
        },
    },
    {
        "type": "function",
        "name": "send_email",
        "description": (
            "수신자에게 트랜잭션 이메일 1통을 직접 발송한다. to(이메일 또는 고객 식별자), "
            "subject, body 가 모두 필수이며 SMTP 프로바이더를 통해 즉시 전송 큐에 넣는다. "
            "채널 선택형 범용 알림이 아니라 이메일 채널 전용 저수준 전송 도구다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "수신 이메일 주소 또는 고객 식별자",
                },
                "subject": {
                    "type": "string",
                    "description": "이메일 제목",
                },
                "body": {
                    "type": "string",
                    "description": "이메일 본문",
                },
            },
            "required": ["to", "subject", "body"],
        },
    },
    {
        "type": "function",
        "name": "create_ticket",
        "description": (
            "고객 문의를 처리하기 위한 상담 티켓을 생성한다. customer_id 와 subject 가 "
            "필수이고 priority 를 지정하지 않으면 normal 로 잡힌다. 반환값은 ticket_id, "
            "상태(open), 배정 큐를 포함하며 알림 발송이나 환불 같은 실제 조치는 수행하지 "
            "않는다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "티켓을 생성할 고객 식별자",
                },
                "subject": {
                    "type": "string",
                    "description": "티켓 제목 / 문의 요지",
                },
                "priority": {
                    "type": "string",
                    "description": "우선순위 (선택: low / normal / high)",
                },
            },
            "required": ["customer_id", "subject"],
        },
    },
    {
        "type": "function",
        "name": "get_shipping_status",
        "description": (
            "한 주문의 배송 상태를 order_id 로 조회한다. 반환값에는 shipment_id, 택배사, "
            "운송장 번호, 현재 배송 상태, 예상 도착일이 포함된다. 결제·청구 상태가 아니라 "
            "물류 배송 단계를 추적하는 용도다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "배송 상태를 조회할 주문 식별자",
                },
            },
            "required": ["order_id"],
        },
    },
]


IMPL = {
    "search_users": _conf_search_users,
    "get_account": _conf_get_account,
    "get_customer_profile": _conf_get_customer_profile,
    "list_invoices": _conf_list_invoices,
    "list_transactions": _conf_list_transactions,
    "fetch_payments": _conf_fetch_payments,
    "cancel_charge": _conf_cancel_charge,
    "reverse_transaction": _conf_reverse_transaction,
    "notify_user": _conf_notify_user,
    "send_email": _conf_send_email,
    "create_ticket": _conf_create_ticket,
    "get_shipping_status": _conf_get_shipping_status,
}

# ── 혼동 그룹 캡처 ──
TOOLS_C, IMPL_C = TOOLS, IMPL
print("혼동 노이즈 도구:", [t["name"] for t in TOOLS_C])


## 네임스페이스 병합 + 디스패처

In [ ]:
TOOLS_FULL = TOOLS_REL + TOOLS_TRAP + TOOLS_C
TOOLS_CLEAN = TOOLS_REL
IMPL = {}
for _d in (IMPL_REL, IMPL_TRAP, IMPL_C):
    IMPL.update(_d)
RELEVANT_NAMES = {t["name"] for t in TOOLS_REL}
TRAP_NAMES = {t["name"] for t in TOOLS_TRAP}
assert len(TOOLS_FULL) == 20 and len(IMPL) == 20, (len(TOOLS_FULL), len(IMPL))

def run_tool(name, args):
    fn = IMPL.get(name)
    if fn is None:
        return f"<error>알 수 없는 도구: {name}</error>"
    try:
        return fn(args)
    except Exception as e:
        return f"<error>{type(e).__name__}: {e}</error>"

print(f"네임스페이스: 전체 {len(TOOLS_FULL)} (정답 {len(TOOLS_REL)} + 함정 {len(TOOLS_TRAP)} + 노이즈 {len(TOOLS_C)}) · 클린 {len(TOOLS_CLEAN)}")

## 평가 과제 (엄격 검증) — 함정을 고르면 실패한다

must_exclude 로 함정 흔적(과환불 RFND-CHG-2, 주문통째환불 RO-ORD-1002)을 잡는다.

In [ ]:
@dataclass
class Task:
    id: str
    steps: int
    prompt: str
    must_include: list
    must_exclude: list = field(default_factory=list)

    def verify(self, answer):
        low = (answer or "").lower().replace(",", "")
        missing = [s for s in self.must_include if s.lower().replace(",", "") not in low]
        wrong = [s for s in self.must_exclude if s.lower().replace(",", "") in low]
        if missing:
            return False, "누락: " + ", ".join(missing)
        if wrong:
            return False, "함정 흔적 포함: " + ", ".join(wrong)
        return True, "정답 요소 충족"

EVAL_TASKS = [
    Task("N1", 2, "고객 sarah@example.com 의 'paid' 상태 주문들의 총액 합(원)을 구하라.", ["57000"]),
    Task("N2", 3, "고객 sarah@example.com 의 주문 ORD-1002 에 중복 청구(같은 금액 succeeded 2건 이상)가 있는지 확인하고, 가장 먼저 성공한 1건은 정상으로 두고 나머지 중복 charge_id 를 모두 답하라.", ["CHG-3", "CHG-4"]),
    Task("N3", 5, "고객 sarah@example.com 이 주문 ORD-1002 에 3번(CHG-2·CHG-3·CHG-4) 청구됐다. 정상 1건인 가장 먼저 성공한 CHG-2 는 그대로 두고 중복분만 환불한 뒤, 고객에게 이메일로 알리고 환불 확인 id 들과 알림 id 를 답하라.",
         ["RFND-CHG-3", "RFND-CHG-4"], ["RFND-CHG-2", "RO-ORD-1002"]),
    Task("N4", 2, "고객 minjun@example.com 의 취소된(cancelled) 주문의 금액(원)을 알려줘.", ["30000"]),
]
print("과제", len(EVAL_TASKS), "개 (엄격)")

## 실행 하네스 — 오호출·함정호출 카운트

In [ ]:
SYSTEM_PROMPT = (
    "너는 전자상거래 고객지원 운영 에이전트다. 제공된 도구만으로 요청을 처리하라. "
    "도구 설명을 꼼꼼히 읽고 목적에 정확히 맞는 도구를 골라라(비슷해 보여도 동작이 다른 도구가 있다). "
    "필요한 정보는 반드시 도구로 조회한 뒤 결론을 내려라(추측 금지).\n"
    "1) 도구 호출 앞에 <plan>...</plan> 으로 다음 스텝을 한 줄 적어라.\n"
    "2) 마지막에 <answer>...</answer> 로 최종 답(요청된 id·금액 포함)을 적어라.\n"
)

def run_task(client, task, tools, arm="", model=None, max_turns=15):
    model = model or MODEL
    input_list = [{"role": "user", "content": task.prompt}]
    called = []
    n_calls = n_err = in_tok = out_tok = turns = 0
    final = ""
    t0 = time.time()
    for _ in range(max_turns):
        resp = client.responses.create(model=model, instructions=SYSTEM_PROMPT,
                                       input=input_list, tools=tools, parallel_tool_calls=True)
        turns += 1
        if resp.usage:
            in_tok += resp.usage.input_tokens
            out_tok += resp.usage.output_tokens
        input_list += resp.output
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            final = resp.output_text
            break
        for c in calls:
            try:
                args = json.loads(c.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            res = run_tool(c.name, args)
            n_calls += 1
            called.append(c.name)
            if res.startswith("<error>"):
                n_err += 1
            input_list.append({"type": "function_call_output", "call_id": c.call_id, "output": res})
    else:
        final = final or "(max_turns 도달)"
    passed, reason = task.verify(final)
    wrong = [n for n in called if n not in RELEVANT_NAMES]
    traps = [n for n in called if n in TRAP_NAMES]
    return {"arm": arm, "task_id": task.id, "passed": passed, "reason": reason, "turns": turns,
            "tool_calls": n_calls, "tool_errors": n_err, "wrong_tool_calls": len(wrong), "trap_calls": len(traps),
            "wrong_names": sorted(set(wrong)), "trap_names": sorted(set(traps)),
            "input_tokens": in_tok, "output_tokens": out_tok, "duration_s": round(time.time() - t0, 2),
            "called": called, "final_text": final}

## A/B 실행 — full(20) vs clean(5), TRIALS=3

혼동·함정이 정확도·오호출·함정호출에 주는 영향을 본다. (24 루프, 멀티턴 — 시간 소요)

In [ ]:
ARMS = [("full(20: 정답5+함정3+노이즈12)", TOOLS_FULL), ("clean(정답5만)", TOOLS_CLEAN)]
TRIALS = 3

if client:
    allr = []
    for label, tools in ARMS:
        rs = [run_task(client, t, tools, label) for t in EVAL_TASKS for _ in range(TRIALS)]
        allr += rs
        n = len(rs)
        print(f"{label:30} 정확도 {sum(r['passed'] for r in rs)/n:4.0%} · 오호출 {sum(r['wrong_tool_calls'] for r in rs)} · "
              f"함정호출 {sum(r['trap_calls'] for r in rs)} · 평균 입력토큰 {sum(r['input_tokens'] for r in rs)/n:,.0f}")
    print("\n과제별 (full):")
    for t in EVAL_TASKS:
        fu = [r for r in allr if r['arm'] == ARMS[0][0] and r['task_id'] == t.id]
        cl = [r for r in allr if r['arm'] == ARMS[1][0] and r['task_id'] == t.id]
        tn = sorted({n for r in fu for n in r['trap_names']})
        print(f"  {t.id} | full {sum(r['passed'] for r in fu)}/{len(fu)} 함정{sum(r['trap_calls'] for r in fu)}{(' '+str(tn)) if tn else ''} | clean {sum(r['passed'] for r in cl)}/{len(cl)}")
    outdir = pathlib.Path.cwd() / "results"; outdir.mkdir(exist_ok=True)
    out = outdir / f"confusable-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    out.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in allr))
    print(f"\nTRIALS={TRIALS} · 저장:", out)
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

## 구조 분석 (API 불필요) — 함정이 왜 오답을 만드는지

In [ ]:
print("정답:", sorted(RELEVANT_NAMES))
print("함정:", sorted(TRAP_NAMES))
print("노이즈:", [t["name"] for t in TOOLS_C])
print()
print("함정이 만드는 잘못된 결과(정답 도구와 대조):")
print(" [N1] list_orders(paid)=정답  vs  get_order_history(stale):")
import json as _J
correct_paid = [o for o in SUPPORT_DATA["orders"] if o["customer_id"]=="CUST-1" and o["status"]=="paid"]
print("     정답 paid합:", sum(o["total"] for o in correct_paid), "(57000)  |  함정:", run_tool("get_order_history", {"customer_id":"CUST-1"}))
print(" [N2] get_charges(ORD-1002)=정답  vs  get_payment_summary(중복 숨김):")
print("     정답:", run_tool("get_charges", {"order_id":"ORD-1002"})[:80])
print("     함정:", run_tool("get_payment_summary", {"order_id":"ORD-1002"}))
print(" [N3] issue_refund(CHG-3)=정답  vs  refund_order(과환불):")
print("     함정:", run_tool("refund_order", {"order_id":"ORD-1002","reason":"x"}))

## 결론 — 헷갈리는 네임스페이스 & 함정

- **혼동·함정이 있으면 정확도가 실제로 갈릴 수 있다**: 도메인이 확연히 다른 방해(앞 노트북)와 달리, 같은 도메인 near-duplicate + 동작이 틀린 함정은 모델이 **잘못된 도구를 골라 오답**을 낼 여지를 만든다. 위 A/B 의 full 정확도·함정호출(trap_calls)이 그 크기다.
- **이름·설명이 방어선이다**: 함정 설명은 정직하게 '올바른 도구를 쓰라'고 안내한다. 모델이 설명을 잘 읽으면 피하고, 대충 이름만 보면 걸린다. → **도구 이름을 명확히 구분되게 짓고, 설명에 '언제 쓰지 말지'까지 적어라**(Anthropic 가이드: 좋은 네이밍·설명이 오호출을 줄인다).
- **검증기는 엄격해야 진실이 보인다**: 함정 도구는 그럴듯한 답(RO- 환불 id, 요약 총액)을 내므로, 느슨한 검증기는 통과시켜 버린다. must_exclude 로 함정 흔적을 잡아야 실제 오답이 드러난다(앞 노트북의 과환불 거짓통과 교훈).
- **근본 해법은 네임스페이스 축소**: clean(5) 는 애초에 함정이 없어 흔들리지 않는다. 실전에선 ToolSearch/deferred 로 그 순간 관련 도구만 올려 '헷갈릴 도구 자체를 안 보이게' 하는 게 가장 강력한 방어다.

> 혼동 노이즈 12개는 하위 서브에이전트가 작성한 같은 도메인 near-duplicate, 함정 3개는 동작을 틀리게 직접 설계했다.